# Employee Attrition Prediction System

## 1. Business Problem

The HR department is facing increasing employee attrition.

High attrition results in:
- Increased hiring costs
- Higher training expenses
- Productivity loss

The company wants to:
- Identify employees likely to leave
- Understand key drivers behind attrition
- Take preventive action

## 2. Project Objectives

• Predict employee attrition (Yes/No)  
• Identify key factors influencing attrition  
• Provide business recommendations  
• Compare multiple ML models  

# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, roc_curve

# Load Dataset

In [2]:
df = pd.read_csv("/content/WA_Fn-UseC_-HR-Employee-Attrition.csv")

In [3]:
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


# Dataset Overview

In [4]:
df.shape

(1470, 35)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.00,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.00,3.0,4.00,5.0
EmployeeCount,1470.0,1.000000,0.000000,1.0,1.00,1.0,1.00,1.0
EmployeeNumber,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.00,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.00,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.00,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.00,2.0,3.00,5.0


# Check Missing Values

In [7]:
df.isnull().sum()

,0
Age,0
Attrition,0
BusinessTravel,0
DailyRate,0
Department,0
DistanceFromHome,0
Education,0
EducationField,0
EmployeeCount,0
EmployeeNumber,0


# Imputation

In [8]:
num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

/tmp/ipython-input-832047620.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipython-input-832047620.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 

# Remove Duplicates

In [9]:
df = df.drop_duplicates()

# Encode Target

In [10]:
df["Attrition"] = df["Attrition"].map({"Yes":1, "No":0})

# Encode Categoricals

In [11]:
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

# Outlier Detection (IQR Example)

In [12]:
Q1 = df["MonthlyIncome"].quantile(0.25)
Q3 = df["MonthlyIncome"].quantile(0.75)
IQR = Q3 - Q1

df = df[~((df["MonthlyIncome"] < (Q1 - 1.5 * IQR)) |
          (df["MonthlyIncome"] > (Q3 + 1.5 * IQR)))]

# Feature Scaling

In [13]:
X = df.drop("Attrition", axis=1)
y = df["Attrition"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# EXPLORATORY DATA ANALYSIS

# Attrition Distribution

In [14]:
fig = px.histogram(df, x="Attrition", color="Attrition",
                   title="Attrition Distribution")
fig.show()

# Salary vs Attrition

In [15]:
fig = px.box(df, x="Attrition", y="MonthlyIncome",
             color="Attrition",
             title="Salary vs Attrition")
fig.show()

# Job Satisfaction vs Attrition

In [16]:
fig = px.box(df, x="Attrition", y="JobSatisfaction",
             color="Attrition",
             title="Job Satisfaction vs Attrition")
fig.show()

# Correlation Heatmap

In [17]:
corr = df.corr()
fig = px.imshow(corr, text_auto=True, title="Correlation Heatmap")
fig.show()

# STATISTICAL ANALYSIS

# Mean, Median, Std

In [18]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1356.0,36.038348,8.808835,18.0,30.00,35.0,41.00,60.0
Attrition,1356.0,0.171091,0.376728,0.0,0.00,0.0,0.00,1.0
BusinessTravel,1356.0,1.601770,0.667073,0.0,1.00,2.0,2.00,2.0
DailyRate,1356.0,802.220501,403.498941,102.0,465.75,804.0,1157.00,1499.0
Department,1356.0,1.270649,0.526544,0.0,1.00,1.0,2.00,2.0
DistanceFromHome,1356.0,9.351032,8.131071,1.0,2.00,7.0,14.00,29.0
Education,1356.0,2.907817,1.026396,1.0,2.00,3.0,4.00,5.0
EducationField,1356.0,2.261799,1.336974,0.0,1.00,2.0,3.00,5.0
EmployeeCount,1356.0,1.000000,0.000000,1.0,1.00,1.0,1.00,1.0
EmployeeNumber,1356.0,1028.637906,605.959746,1.0,490.25,1017.5,1568.25,2068.0


# Hypothesis Testing

In [19]:
from scipy.stats import ttest_ind

group1 = df[df["Attrition"]==1]["MonthlyIncome"]
group2 = df[df["Attrition"]==0]["MonthlyIncome"]

ttest_ind(group1, group2)

TtestResult(statistic=np.float64(-5.251222069610486), pvalue=np.float64(1.752751550329376e-07), df=np.float64(1354.0))

# TRAIN TEST SPLIT

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# MACHINE LEARNING MODELS

# Logistic Regression

In [21]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Decision Tree

In [22]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

# Random Forest

In [23]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# SVM

In [24]:
from sklearn.svm import SVC
svm = SVC(probability=True)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

# KNN

In [25]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

# MODEL EVALUATION

In [26]:
def evaluate(y_true, y_pred):
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1 Score:", f1_score(y_true, y_pred))

# Plot Model Comparison

In [28]:
models = {
    "Logistic Regression": y_pred_lr,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf,
    "SVM": y_pred_svm,
    "KNN": y_pred_knn,
}

results = []
for name, y_pred in models.items():
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({"Model": name, "Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1 Score": f1})

results_df = pd.DataFrame(results)
display(results_df)

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.882353,0.807692,0.437500,0.567568
1,Decision Tree,0.783088,0.382979,0.375000,0.378947
2,Random Forest,0.830882,0.666667,0.083333,0.148148
3,SVM,0.852941,0.900000,0.187500,0.310345
4,KNN,0.834559,0.714286,0.104167,0.181818


In [29]:
fig = px.bar(results_df,
             x="Model",
             y="Accuracy",
             title="Model Accuracy Comparison",
             color="Model")

fig.show()

# Confusion Matrix

In [30]:
cm = confusion_matrix(y_test, y_pred_rf)
fig = px.imshow(cm, text_auto=True, title="Confusion Matrix")
fig.show()

# ROC-AUC Curve

In [31]:
y_prob = rf.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC Curve'))
fig.show()

## Business Insights

Why employees leave:
• Low job satisfaction
• Lower salary levels
• Poor work-life balance
• Fewer years at company


# Most influential features:

• Monthly Income
• Job Satisfaction
• Years at Company
• Work-Life Balance

# HR Strategies:

• Improve employee engagement
• Introduce salary adjustments
• Enhance career growth programs
• Conduct satisfaction surveys regularly